# 10 Tracing
Put a span around each step of the app's answer function, send 20 requests, and read latency and cost from the spans.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-10"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: one JSON line per span

In [2]:
import json, logging
import pandas as pd
from kit.llm import pick_provider
from kit.prices import cost
from kit.tracing import DEFAULT_REQUESTS, Tracer, load_spans, passed, summarize, traced_answer

for name in ("httpx", "httpx2"):  # the MCP library turns on one log line per HTTP call; hide them
    logging.getLogger(name).setLevel(logging.WARNING)

SPANS = OUTPUT / "traces" / "spans.jsonl"
SPANS.unlink(missing_ok=True)
(SPANS.parent / "outbox.jsonl").unlink(missing_ok=True)

# kit/tracing.py wraps app/app.py's steps: request > retrieval > model_call (> tool_call).
tracer = Tracer(SPANS)
traced_answer("How do I export a note as PDF?", tracer, folder=CORPUS)
print("Provider:", pick_provider())
for line in SPANS.read_text().splitlines():
    s = json.loads(line)
    print(f"{s['name']:11} {s['latency_ms']:>8} ms   tokens in/out {s['input_tokens']}/{s['output_tokens']}   ${s['cost_usd']:.6f}")

Provider: mock
retrieval        1.1 ms   tokens in/out 0/0   $0.000000
model_call     617.0 ms   tokens in/out 262/69   $0.000538
request        618.5 ms   tokens in/out 262/69   $0.000538


Step 3: 20 requests, each with a success check

In [3]:
# 17 questions with the page that should be cited, and 3 that ask the app to draft an issue.
# Success: the expected page is cited, or the draft_issue tool call worked.
requests = DEFAULT_REQUESTS
print(len(requests), "requests,", sum(bool(r.get("issue")) for r in requests), "of them ask for a draft issue")
pd.DataFrame(requests).tail(5)

20 requests, 3 of them ask for a draft issue


,question,expect,issue
15,Can our team create templates that everyone in...,qn-12,NaN
16,How can I organize notes with tags and then fi...,qn-03,NaN
17,PDF export cuts off wide tables. Draft an issu...,NaN,True
18,"Sync fails on Android. Draft an issue titled ""...",NaN,True
19,Dark mode resets after a restart. Draft an iss...,NaN,True


Step 4: send them, one trace each

In [4]:
SPANS.unlink(missing_ok=True)
(SPANS.parent / "outbox.jsonl").unlink(missing_ok=True)
tracer = Tracer(SPANS)
results = [(req, traced_answer(req["question"], tracer, folder=CORPUS)) for req in requests]
spans = load_spans(SPANS)
success = {r["trace_id"]: passed(req, r, spans) for req, r in results}
print(len(spans), "spans in traces/spans.jsonl:", pd.Series([s["name"] for s in spans]).value_counts().to_dict())
print("Successful tasks:", sum(success.values()), "of", len(requests))

63 spans in traces/spans.jsonl: {'retrieval': 20, 'model_call': 20, 'request': 20, 'tool_call': 3}
Successful tasks: 19 of 20


Step 5: p50 and p95 latency, and time to first token

In [5]:
summary = summarize(spans, success)
NOTES = {
    "stream": "TTFT is the first streamed text chunk.",
    "first byte (not streamed)": "this call was not streamed, so the first byte arrives with the whole answer: TTFT equals the full model time.",
    "simulated stream": "offline mock: it does not stream, so its TTFT is made up (30 percent of its made up latency).",
}
print(f"Latency  p50 {summary['p50_latency_ms']:.0f} ms   p95 {summary['p95_latency_ms']:.0f} ms")
print(f"TTFT     p50 {summary['p50_ttft_ms']:.0f} ms   p95 {summary['p95_ttft_ms']:.0f} ms")
for kind in summary["ttft_kind"]:
    print("Note:", NOTES.get(kind, kind))
print("With 20 requests, p95 is set by the one or two slowest. Send more before you trust it.")

Latency  p50 644 ms   p95 971 ms
TTFT     p50 194 ms   p95 292 ms
Note: offline mock: it does not stream, so its TTFT is made up (30 percent of its made up latency).
With 20 requests, p95 is set by the one or two slowest. Send more before you trust it.


Step 6: cost per request and cost per successful task

In [6]:
print(f"Total: ${summary['cost_total_usd']:.5f} for {summary['requests']} requests")
print(f"Cost per request:         ${summary['cost_per_request_usd']:.6f}")
print(f"Cost per successful task: ${summary['cost_per_successful_task_usd']:.6f} "
      f"({summary['successes']} of {summary['requests']} succeeded)")
print("Failed requests still cost money, so cost per successful task is the higher, honest number.")

Total: $0.01149 for 20 requests
Cost per request:         $0.000574
Cost per successful task: $0.000604 (19 of 20 succeeded)
Failed requests still cost money, so cost per successful task is the higher, honest number.


Step 7: the span that drives spend

In [7]:
df = pd.DataFrame(spans)
steps = df[df.name != "request"]
by_span = steps.groupby("name").agg(spans=("span_id", "count"), cost_usd=("cost_usd", "sum"),
                                    p50_ms=("latency_ms", "median"))
by_span["share_of_cost"] = (by_span.cost_usd / by_span.cost_usd.sum()).round(3)
print(by_span.sort_values("cost_usd", ascending=False), "\n")

top = steps.sort_values("cost_usd", ascending=False).iloc[0]
question = df[(df.trace_id == top.trace_id) & (df.name == "request")]["attrs"].iloc[0]["question"]
input_share = cost(int(top.input_tokens), 0, top.model, int(top.cached_tokens)) / top.cost_usd
print("Most expensive single span:", top["name"], "|", top.model, "|", int(top.input_tokens), "in,",
      int(top.output_tokens), "out | $", round(float(top.cost_usd), 6))
print("Its request:", question)
print(f"Input tokens (the help pages in the prompt) are {input_share:.0%} of that span's cost, output tokens the rest.")

# Save the evidence next to spans.jsonl: the numbers above, and the spend by span.
by_span.sort_values("cost_usd", ascending=False).to_csv(OUTPUT / "span_costs.csv")
keep = ["requests", "successes", "p50_latency_ms", "p95_latency_ms", "p50_ttft_ms", "p95_ttft_ms", "ttft_kind",
        "cost_total_usd", "cost_per_request_usd", "cost_per_successful_task_usd"]
saved = {k: summary[k] for k in keep if k in summary} | {
    "top_spend_span": by_span.cost_usd.idxmax(), "top_spend_span_share": float(by_span.share_of_cost.max()),
    "provider": pick_provider()}
(OUTPUT / "tracing_summary.json").write_text(json.dumps(saved, indent=1, default=str))
print("\nSaved tracing_summary.json and span_costs.csv:", json.dumps(saved, default=str)[:300])

            spans  cost_usd  p50_ms  share_of_cost
name                                              
model_call     20  0.011485   643.0            1.0
retrieval      20  0.000000     0.8            0.0
tool_call       3  0.000000     0.1            0.0 

Most expensive single span: model_call | mock-model | 268 in, 87 out | $ 0.000616
Its request: Where can I see the different plans and subscription prices?
Input tokens (the help pages in the prompt) are 44% of that span's cost, output tokens the rest.

Saved tracing_summary.json and span_costs.csv: {"requests": 20, "successes": 19, "p50_latency_ms": 643.9, "p95_latency_ms": 970.8, "p50_ttft_ms": 193.8, "p95_ttft_ms": 292.05, "ttft_kind": ["simulated stream"], "cost_total_usd": 0.011485000000000002, "cost_per_request_usd": 0.0005742500000000001, "cost_per_successful_task_usd": 0.000604473684210


Optional: send the same spans to Langfuse

In [8]:
# Optional. Skipped unless LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY are set; nothing above needs it.
# Langfuse's public ingestion API takes a batch of events: a trace per request, a generation per model
# call (with token usage) and a span for every other step. Not tested against a real Langfuse account.
import uuid
from datetime import datetime, timedelta
from kit.llm import get_secret, offline

def end_time(s):
    return (datetime.fromisoformat(s["start"]) + timedelta(milliseconds=s["latency_ms"])).isoformat()

def to_langfuse(spans):
    events = []
    for s in spans:
        if s["name"] == "request":
            kind, body = "trace-create", {"id": s["trace_id"], "name": "help-answer", "timestamp": s["start"],
                                          "input": s["attrs"].get("question"), "metadata": s["attrs"]}
        else:
            kind = "generation-create" if s["name"] == "model_call" else "span-create"
            body = {"id": s["span_id"], "traceId": s["trace_id"], "name": s["name"], "startTime": s["start"],
                    "endTime": end_time(s), "metadata": s["attrs"]}
            if s["name"] == "model_call":
                body.update(model=s["model"], usage={"input": s["input_tokens"], "output": s["output_tokens"]})
        events.append({"id": uuid.uuid4().hex, "timestamp": s["start"], "type": kind, "body": body})
    return events

batch = to_langfuse(spans)
print(len(batch), "events. One generation event:")
print(json.dumps(next(e for e in batch if e["type"] == "generation-create"), indent=1)[:700])
public, secret = get_secret("LANGFUSE_PUBLIC_KEY"), get_secret("LANGFUSE_SECRET_KEY")
if public and secret and not offline():
    import requests
    host = get_secret("LANGFUSE_HOST") or "https://cloud.langfuse.com"
    resp = requests.post(f"{host}/api/public/ingestion", json={"batch": batch}, auth=(public, secret), timeout=30)
    print("Langfuse replied", resp.status_code)
else:
    print("\nNo Langfuse keys (or offline): nothing was sent.")

63 events. One generation event:
{
 "id": "f1648cc096d04ba5a1e8864d10480c32",
 "timestamp": "2026-11-15T11:18:39.486+00:00",
 "type": "generation-create",
 "body": {
  "id": "237896cf",
  "traceId": "d0e68e998ba5",
  "name": "model_call",
  "startTime": "2026-11-15T11:18:39.486+00:00",
  "endTime": "2026-11-15T11:18:40.378000+00:00",
  "metadata": {
   "provider": "mock",
   "ttft_kind": "simulated stream",
   "stop_reason": "STOP",
   "cached_reported": true
  },
  "model": "mock-model",
  "usage": {
   "input": 265,
   "output": 72
  }
 }
}

No Langfuse keys (or offline): nothing was sent.
